# 09 — CNN rung 4, experiment 1: family-oversampling

**Decision this feeds** (`RESOURCES.md`'s Wenzel et al. 2019 finding,
adapted to our own evidence — see `README.md`/chat 2026-09-09): rung 3's
own per-family log loss breakdown found the CNN's two worst-performing
families are **2.46mm (n=528, log loss 0.5191) and 3.895mm (n=325, log
loss 0.5035)** — notably, these are the two *largest* families, not
underrepresented ones, so Wenzel et al.'s literal "oversample the
underrepresented coarse family" recipe doesn't directly apply. This
experiment tests a version grounded in our own data instead:
`torch.utils.data.WeightedRandomSampler` boosting exactly those two
families during training (`src/evaluate.py::family_oversample_weights`),
nothing else changed.

**Gate**: this experiment is scored against the **current validated
CNN** (rung 3, `README.md` 2026-09-09: mean=0.4520, sd=0.0109), not the
classical baseline — it either replaces the current CNN or it doesn't.
Same nested-CV + paired-bootstrap protocol as
`notebooks/07_cnn_rung3.ipynb`.

**Timing**: `notebooks/08_training_throughput_check.ipynb` measured
~1.5s/epoch on a full-size fold and confirmed the volume cache is
reused (this experiment doesn't touch preprocessing) — a full 5×5
nested-CV run here costs roughly 7-18 minutes, not hours.

**Data handling**: this notebook loads real `.nii.gz` volumes and
row-level labels throughout, so per the AI-assistant data rule
(`README.md`) it is **[RUN ME]** — run it yourself, share back only the
printed aggregate numbers, never any per-row output.

In [1]:
# [RUN ME] -- loads real pixel data + row-level labels. Rebuilds (or
# reuses -- see notebook 08) the shared on-disk volume cache.
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import torch

import cache
import config
import dataset
import evaluate
import model
import train as train_mod

labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
family_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")[
    [config.UID_COLUMN, "inplane_family"]
]
labeled_df = labels_df.merge(family_df, on=config.UID_COLUMN, how="inner").reset_index(drop=True)

uids = labeled_df[config.UID_COLUMN].tolist()
labels = labeled_df[config.TARGET_COLUMN].tolist()
families = labeled_df["inplane_family"].tolist()

config_fingerprint = {
    "TARGET_SPACING": config.TARGET_SPACING,
    "CROP_SIZE_MM": config.CROP_SIZE_MM,
    "CROP_CENTER_MM": config.CROP_CENTER_MM,
    "TARGET_SHAPE": config.TARGET_SHAPE,
    "BACKGROUND_PERCENTILE": config.BACKGROUND_PERCENTILE,
    "BACKGROUND_MAX_FRACTION": config.BACKGROUND_MAX_FRACTION,
}

cache_start = time.time()
volume_cache = cache.CachedVolumeStore(
    uids, cache_dir=config.DATA_PROCESSED / "volume_cache",
    config_fingerprint=config_fingerprint,
)
print(f"cache {'reused' if volume_cache.was_reused else 'rebuilt'} in "
      f"{time.time() - cache_start:.1f}s for {len(uids)} volumes")

cache reused in 0.2s for 1362 volumes


In [2]:
# [RUN ME] (no data access itself). Same helper as notebooks 06/07, plus
# optional family-oversampling on the INNER-TRAIN split only -- never on
# inner-val or the outer test rows, since oversampling changes how often
# a row is seen during training, not what it's scored against.
def train_and_score_nested(train_uids, train_labels, train_family,
                            outer_uids, batch_size, lr, seed,
                            boosted_families=(), boost_factor=1.0,
                            epochs=config.EPOCHS, patience=config.PATIENCE,
                            inner_splits=10):
    inner_train_idx, inner_val_idx = evaluate.make_folds(
        train_labels, train_family, n_splits=inner_splits, random_state=seed
    )[0]

    def subset(idxs):
        return ([train_uids[i] for i in idxs], [train_labels[i] for i in idxs],
                 [train_family[i] for i in idxs])

    inner_train_uids, inner_train_labels, inner_train_family = subset(inner_train_idx)
    inner_val_uids, inner_val_labels, _ = subset(inner_val_idx)

    inner_train_ds = dataset.DatParkinsonDataset(inner_train_uids, inner_train_labels, load_fn=volume_cache.get)
    inner_val_ds = dataset.DatParkinsonDataset(inner_val_uids, inner_val_labels, load_fn=volume_cache.get)

    if boosted_families:
        weights = evaluate.family_oversample_weights(inner_train_family, boosted_families, boost_factor)
        sampler = torch.utils.data.WeightedRandomSampler(
            torch.as_tensor(weights, dtype=torch.double), num_samples=len(weights), replacement=True)
        inner_train_loader = torch.utils.data.DataLoader(inner_train_ds, batch_size=batch_size, sampler=sampler, num_workers=0)
    else:
        inner_train_loader = torch.utils.data.DataLoader(inner_train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    inner_val_loader = torch.utils.data.DataLoader(inner_val_ds, batch_size=batch_size, num_workers=0)

    torch.manual_seed(seed)
    net = model.build_model().to(config.DEVICE)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=config.WEIGHT_DECAY)
    loss_fn = torch.nn.BCEWithLogitsLoss()

    best_state, history = train_mod.train_one_fold(
        net, inner_train_loader, inner_val_loader, optimizer, loss_fn,
        epochs=epochs, patience=patience, device=config.DEVICE,
        use_amp=config.USE_AMP, seed=seed,
    )
    net.load_state_dict(best_state)

    outer_ds = dataset.DatParkinsonDataset(outer_uids, load_fn=volume_cache.get)
    outer_loader = torch.utils.data.DataLoader(outer_ds, batch_size=batch_size, num_workers=0)
    outer_probs = []
    for x, _ in outer_loader:
        outer_probs.append(model.predict(net, x))
    return np.concatenate(outer_probs), history, best_state

In [3]:
# [RUN ME] -- mini-experiment: 3 boost_factor candidates on outer fold 0
# only (same pattern as notebook 06's batch/LR mini-experiment). Picks
# the winner by inner-validation log loss before committing to the full
# 5x5 nested-CV gate below. batch_size/lr held fixed at rung 2/3's
# validated winner (32, 2e-3) -- not re-tuned here, this experiment only
# varies the sampler.
BOOSTED_FAMILIES = ("2.46", "3.895")
batch_size, lr = 32, 2e-3

outer_folds = evaluate.make_folds(np.array(labels), np.array(families),
                                   n_splits=config.N_FOLDS, random_state=config.SEED)
fold0_train_idx, fold0_test_idx = outer_folds[0]
fold0_train_uids = [uids[i] for i in fold0_train_idx]
fold0_train_labels = [labels[i] for i in fold0_train_idx]
fold0_train_family = [families[i] for i in fold0_train_idx]
fold0_test_uids = [uids[i] for i in fold0_test_idx]
fold0_test_labels = np.array([labels[i] for i in fold0_test_idx])

candidates = [1.5, 2.0, 3.0]
boost_results = {}
for boost_factor in candidates:
    start = time.time()
    probs, history, best_state = train_and_score_nested(
        fold0_train_uids, fold0_train_labels, fold0_train_family,
        fold0_test_uids, batch_size=batch_size, lr=lr, seed=config.SEED,
        boosted_families=BOOSTED_FAMILIES, boost_factor=boost_factor,
    )
    elapsed = time.time() - start
    score = evaluate.log_loss_score(fold0_test_labels, probs)
    boost_results[boost_factor] = {"history": history, "score": score, "seconds": elapsed}
    print(f"boost_factor={boost_factor}: inner-val best={min(history['val_loss']):.4f}, "
          f"outer log loss={score:.4f}, {elapsed:.1f}s ({elapsed / len(history['val_loss']):.2f}s/epoch)")

winner_boost_factor = min(boost_results, key=lambda k: min(boost_results[k]["history"]["val_loss"]))
print(f"\nwinner (lowest inner-validation log loss): boost_factor={winner_boost_factor}")

boost_factor=1.5: inner-val best=0.4036, outer log loss=0.4521, 16.0s (1.00s/epoch)
boost_factor=2.0: inner-val best=0.3575, outer log loss=0.3403, 25.3s (0.82s/epoch)
boost_factor=3.0: inner-val best=0.4224, outer log loss=0.4549, 11.7s (0.83s/epoch)

winner (lowest inner-validation log loss): boost_factor=2.0


In [4]:
# [RUN ME] -- full 5-fold nested CV, repeated 5x, with the winning
# boost_factor from the mini-experiment above. Same protocol as rung 3
# (notebooks/07_cnn_rung3.ipynb, model-init seed and fold-split seed
# varied together per Bouthillier et al. 2021).
N_REPEATS = 5
oof_repeats_boosted = []

for repeat_seed in range(config.SEED, config.SEED + N_REPEATS):
    outer_folds = evaluate.make_folds(np.array(labels), np.array(families),
                                       n_splits=config.N_FOLDS, random_state=repeat_seed)
    oof_probs = np.zeros(len(uids))
    for fold_i, (train_idx, test_idx) in enumerate(outer_folds):
        fold_train_uids = [uids[i] for i in train_idx]
        fold_train_labels = [labels[i] for i in train_idx]
        fold_train_family = [families[i] for i in train_idx]
        fold_test_uids = [uids[i] for i in test_idx]

        probs, history, best_state = train_and_score_nested(
            fold_train_uids, fold_train_labels, fold_train_family,
            fold_test_uids, batch_size=batch_size, lr=lr, seed=repeat_seed,
            boosted_families=BOOSTED_FAMILIES, boost_factor=winner_boost_factor,
        )
        oof_probs[test_idx] = probs
        torch.save(best_state, config.CHECKPOINT_DIR / f"rung4_familybias_seed{repeat_seed}_fold{fold_i}.pt")
        fold_score = evaluate.log_loss_score(np.array(labels)[test_idx], probs)
        print(f"  seed={repeat_seed} fold={fold_i}: {len(history['val_loss'])} epochs, "
              f"outer fold log loss={fold_score:.4f}")

    repeat_logloss = evaluate.log_loss_score(np.array(labels), oof_probs)
    oof_repeats_boosted.append(oof_probs)
    print(f"seed={repeat_seed} pooled OOF log loss: {repeat_logloss:.4f}")
    np.save(config.DATA_PROCESSED / f"rung4_familybias_oof_seed{repeat_seed}.npy", oof_probs)

repeat_scores_boosted = np.array([evaluate.log_loss_score(np.array(labels), oof) for oof in oof_repeats_boosted])
print(f"\n{N_REPEATS}-repeat family-boosted CNN pooled log loss: "
      f"mean={repeat_scores_boosted.mean():.4f}, sd={repeat_scores_boosted.std(ddof=1):.4f}")
print("current validated CNN (rung 3, README.md 2026-09-09): mean=0.4520, sd=0.0109")

  seed=42 fold=0: 31 epochs, outer fold log loss=0.3403
  seed=42 fold=1: 35 epochs, outer fold log loss=0.4728
  seed=42 fold=2: 13 epochs, outer fold log loss=0.5063
  seed=42 fold=3: 22 epochs, outer fold log loss=0.4907
  seed=42 fold=4: 28 epochs, outer fold log loss=0.4429
seed=42 pooled OOF log loss: 0.4506
  seed=43 fold=0: 26 epochs, outer fold log loss=0.4754
  seed=43 fold=1: 28 epochs, outer fold log loss=0.4409
  seed=43 fold=2: 28 epochs, outer fold log loss=0.4285
  seed=43 fold=3: 24 epochs, outer fold log loss=0.4395
  seed=43 fold=4: 17 epochs, outer fold log loss=0.4934
seed=43 pooled OOF log loss: 0.4555
  seed=44 fold=0: 27 epochs, outer fold log loss=0.4052
  seed=44 fold=1: 26 epochs, outer fold log loss=0.4991
  seed=44 fold=2: 18 epochs, outer fold log loss=0.3915
  seed=44 fold=3: 16 epochs, outer fold log loss=0.4943
  seed=44 fold=4: 15 epochs, outer fold log loss=0.5108
seed=44 pooled OOF log loss: 0.4602
  seed=45 fold=0: 29 epochs, outer fold log loss=0.4

In [ ]:
# [RUN ME] -- control run: boost_factor=1.0 with BOOSTED_FAMILIES still
# set, so the WeightedRandomSampler(replacement=True) code path is
# exercised with uniform (1.0) weights -- i.e. with-replacement bootstrap
# resampling each epoch, but no actual family boosting. Isolates that
# sampler-regime effect from the boosted run above (Opus review,
# 2026-09-10: rung-3's baseline used shuffle=True with no replacement, so
# without this control the boosted run's result conflates "boosting the
# two families" with "switching to with-replacement sampling" -- see
# project memory project_dat_parkinson_rung4_gate_review.md).
oof_repeats_control = []

for repeat_seed in range(config.SEED, config.SEED + N_REPEATS):
    outer_folds = evaluate.make_folds(np.array(labels), np.array(families),
                                       n_splits=config.N_FOLDS, random_state=repeat_seed)
    oof_probs = np.zeros(len(uids))
    for fold_i, (train_idx, test_idx) in enumerate(outer_folds):
        fold_train_uids = [uids[i] for i in train_idx]
        fold_train_labels = [labels[i] for i in train_idx]
        fold_train_family = [families[i] for i in train_idx]
        fold_test_uids = [uids[i] for i in test_idx]

        probs, history, best_state = train_and_score_nested(
            fold_train_uids, fold_train_labels, fold_train_family,
            fold_test_uids, batch_size=batch_size, lr=lr, seed=repeat_seed,
            boosted_families=BOOSTED_FAMILIES, boost_factor=1.0,
        )
        oof_probs[test_idx] = probs
        fold_score = evaluate.log_loss_score(np.array(labels)[test_idx], probs)
        print(f"  seed={repeat_seed} fold={fold_i}: {len(history['val_loss'])} epochs, "
              f"outer fold log loss={fold_score:.4f}")

    repeat_logloss = evaluate.log_loss_score(np.array(labels), oof_probs)
    oof_repeats_control.append(oof_probs)
    print(f"seed={repeat_seed} pooled OOF log loss: {repeat_logloss:.4f}")
    np.save(config.DATA_PROCESSED / f"rung4_familybias_control_oof_seed{repeat_seed}.npy", oof_probs)

repeat_scores_control = np.array([evaluate.log_loss_score(np.array(labels), oof) for oof in oof_repeats_control])
print(f"\n{N_REPEATS}-repeat control (boost_factor=1.0, sampler-only) pooled log loss: "
      f"mean={repeat_scores_control.mean():.4f}, sd={repeat_scores_control.std(ddof=1):.4f}")

In [ ]:
# [RUN ME] (no data access itself). Gate: paired per-repeat delta vs.
# the current validated CNN, using evaluate.paired_repeat_gate (fixed
# 2026-09-10 per Opus review -- the old gate here used
# paired_bootstrap_ci on a single arbitrarily-chosen repeat plus a noise
# threshold scaled by a single repeat's sd instead of the mean's
# standard error; see project memory
# project_dat_parkinson_rung4_gate_review.md). Also isolates the pure
# family-boost effect from the with-replacement-sampler confound by
# comparing the boosted run against the boost_factor=1.0 control, not
# just against the rung-3 baseline.
y_true = np.array(labels)
repeat_seeds = list(range(config.SEED, config.SEED + N_REPEATS))
current_cnn_oof_by_repeat = [
    np.load(config.DATA_PROCESSED / f"rung3_oof_seed{s}.npy") for s in repeat_seeds
]

deltas_vs_current = [
    evaluate.log_loss_score(y_true, oof_repeats_boosted[i]) - evaluate.log_loss_score(y_true, current_cnn_oof_by_repeat[i])
    for i in range(N_REPEATS)
]
deltas_control_vs_current = [
    evaluate.log_loss_score(y_true, oof_repeats_control[i]) - evaluate.log_loss_score(y_true, current_cnn_oof_by_repeat[i])
    for i in range(N_REPEATS)
]
deltas_boosted_vs_control = [
    evaluate.log_loss_score(y_true, oof_repeats_boosted[i]) - evaluate.log_loss_score(y_true, oof_repeats_control[i])
    for i in range(N_REPEATS)
]

gate_vs_current = evaluate.paired_repeat_gate(deltas_vs_current)
gate_control_vs_current = evaluate.paired_repeat_gate(deltas_control_vs_current)
gate_boosted_vs_control = evaluate.paired_repeat_gate(deltas_boosted_vs_control)

print(f"per-repeat deltas (boosted - current CNN): {[f'{d:+.4f}' for d in deltas_vs_current]}")
print(f"boosted vs. current CNN:              mean={gate_vs_current['mean']:+.4f}, sd={gate_vs_current['sd']:.4f}, "
      f"95% CI=[{gate_vs_current['ci_low']:+.4f}, {gate_vs_current['ci_high']:+.4f}] -> "
      f"{'PASSED' if gate_vs_current['passed'] else 'NOT PASSED'}")
print(f"control (sampler-only) vs. current CNN: mean={gate_control_vs_current['mean']:+.4f}, sd={gate_control_vs_current['sd']:.4f}, "
      f"95% CI=[{gate_control_vs_current['ci_low']:+.4f}, {gate_control_vs_current['ci_high']:+.4f}] -> "
      f"{'PASSED' if gate_control_vs_current['passed'] else 'NOT PASSED'} "
      f"(if this passes/fails the same way as the boosted run above, the sampler regime -- not family boosting -- is driving the result)")
print(f"boosted vs. control (isolates the pure family-boost effect): mean={gate_boosted_vs_control['mean']:+.4f}, sd={gate_boosted_vs_control['sd']:.4f}, "
      f"95% CI=[{gate_boosted_vs_control['ci_low']:+.4f}, {gate_boosted_vs_control['ci_high']:+.4f}] -> "
      f"{'PASSED' if gate_boosted_vs_control['passed'] else 'NOT PASSED'}")

gate_passed = gate_vs_current["passed"]
print(f"\nGATE (boosted vs. current CNN) {'PASSED' if gate_passed else 'NOT PASSED'}: "
      f"{'family-oversampling REPLACES the current CNN.' if gate_passed else 'does not beat the current CNN by more than noise -- keep the current CNN.'}")

family_arr = np.array(families)
print("\nper-family log loss (boosted run, repeat seed=42; boosted families marked):")
for fam in sorted(set(families)):
    mask = family_arr == fam
    if mask.sum() < 5:
        continue
    marker = " <-- boosted this run" if fam in BOOSTED_FAMILIES else ""
    print(f"  {fam:<12} n={int(mask.sum()):>4}  log loss="
          f"{evaluate.log_loss_score(y_true[mask], oof_repeats_boosted[0][mask]):.4f}{marker}")

**What we're looking for:** does boosting the 2.46mm and 3.895mm
families during training (the two `README.md`-documented worst
performers) beat the current validated CNN (0.4520) by more than noise
-- and is that actually a family-boosting effect, or an artifact of
switching to with-replacement sampling (control run, `boost_factor=1.0`)?

**What we found:** *(paste: the 3 boost_factor mini-experiment results
and the winner; the 5-repeat boosted mean/sd and control mean/sd; the
three `paired_repeat_gate` results -- boosted-vs-current,
control-vs-current, boosted-vs-control -- and which GATE lines
PASSED/NOT PASSED; the per-family breakdown, especially whether
2.46mm/3.895mm actually improved)*

**Decision / next step:** *(if the boosted-vs-current gate passed: this
CNN replaces the rung-3 CNN in the submission blend -- re-run the
blend-weight leave-one-repeat-out check from
`notebooks/07_cnn_rung3.ipynb` against this new CNN, since w_cnn=0.70
was tuned for the old one. If not: keep the rung-3 CNN, move on to
experiment 2 [LR schedule, `notebooks/10_cnn_lr_schedule.ipynb`], and
log this as a negative result per the project's standing rule -- note
in the writeup whether the control run implicates the sampler regime
specifically.)*